## CAG(Cache Augumneted Generation) ##

In [3]:
import os
import time
from dotenv import load_dotenv
import tiktoken

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# ============================================================
# SETUP
# ============================================================
groq_api_key = os.getenv("GROQAPIKEY")

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0
)

# ============================================================
# PHASE 1: LOAD DOCUMENT + COUNT TOKENS
# ============================================================
print("=" * 70)
print("PHASE 1: LOAD DOCUMENT")
print("=" * 70)

with open("swatvalley.txt", "r", encoding="utf-8") as f:
    knowledge_base = f.read()

print(f"[SUCCESS] Loaded document: {len(knowledge_base):,} characters")

# Groq models aren't in tiktoken's model list, so use the general-purpose
# cl100k_base encoding as a close approximation of token count.
encoding = tiktoken.get_encoding("cl100k_base")
token_count = len(encoding.encode(knowledge_base))
print(f"[SUCCESS] Approx. token count: {token_count:,} tokens")

context_window = 131_072  # qwen/qwen3-32b context window on Groq
utilization = (token_count / context_window) * 100
print(f"[INFO] Context window usage: {utilization:.1f}% (window: {context_window:,} tokens)")

if token_count < context_window * 0.5:
    print("[SUCCESS] Great candidate for CAG (under 50% of context window).")
elif token_count < context_window:
    print("[INFO] Might work with CAG — leave room for question + answer.")
else:
    print("[WARNING] Too large for CAG on this model. Consider RAG instead.")
print()

# ============================================================
# PHASE 2: CAG QUERY FUNCTION
# ============================================================
print("=" * 70)
print("PHASE 2: CAG QUERY FUNCTION")
print("=" * 70)

cag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following knowledge base.
If the knowledge base does not contain the answer, say "I don't have that information."

Knowledge Base:
{context}

Question: {question}

Answer:"""
)

cag_chain = cag_prompt | llm | StrOutputParser()
print("[SUCCESS] CAG chain created")
print()


def cag_query(question: str, knowledge: str, verbose: bool = True) -> tuple[str, float]:
    """Send the full knowledge base + question to the LLM in one shot (no retrieval)."""
    if verbose:
        print(f"[QUERY] {question}")

    start_time = time.time()
    answer = cag_chain.invoke({"context": knowledge, "question": question})
    latency = time.time() - start_time

    if verbose:
        print(f"[SUCCESS] Answer generated in {latency:.2f}s")
        print(f"[ANSWER] {answer}")
        print()

    return answer, latency


# ============================================================
# TEST: SINGLE QUERY
# ============================================================
print("=" * 70)
print("TEST: CAG QUERY")
print("=" * 70)

answer, latency = cag_query("What is Natural Language Processing?", knowledge_base)

# ============================================================
# TEST: MULTIPLE QUERIES + LATENCY SUMMARY
# ============================================================
print("=" * 70)
print("MULTIPLE QUERIES TEST")
print("=" * 70)

test_questions = [
    "What is tokenization in NLP?",
    "What is the transformer architecture?",
    "How does BERT differ from GPT?",
    "What is attention mechanism?",
]

latencies = []
for i, question in enumerate(test_questions, 1):
    print(f"Query {i}/{len(test_questions)}: {question}")
    print("-" * 70)
    answer, latency = cag_query(question, knowledge_base, verbose=False)
    latencies.append(latency)
    preview = answer[:150] + "..." if len(answer) > 150 else answer
    print(f"[SUCCESS] Answered in {latency:.2f}s")
    print(f"[ANSWER] {preview}")
    print()

print("=" * 70)
print("LATENCY SUMMARY")
print("=" * 70)
for i, latency in enumerate(latencies, 1):
    print(f"Query {i}: {latency:.2f}s")

avg_latency = sum(latencies) / len(latencies)
print(f"\n[INFO] Average latency: {avg_latency:.2f}s")

PHASE 1: LOAD DOCUMENT
[SUCCESS] Loaded document: 13,019 characters
[SUCCESS] Approx. token count: 2,856 tokens
[INFO] Context window usage: 2.2% (window: 131,072 tokens)
[SUCCESS] Great candidate for CAG (under 50% of context window).

PHASE 2: CAG QUERY FUNCTION
[SUCCESS] CAG chain created

TEST: CAG QUERY
[QUERY] What is Natural Language Processing?
[SUCCESS] Answer generated in 1.26s
[ANSWER] <think>
Okay, the user is asking, "What is Natural Language Processing?" Let me check the provided knowledge base to see if there's any information related to this.

Looking through the sections, the knowledge base is about Swat Valley in Pakistan—its geography, travel info, cities, and attractions. There's a lot of detail about the region, but nothing about technology, computer science, or NLP. The closest it gets is mentioning "Gandhara Art" and some historical artifacts, but that's not related to NLP. The user is asking about a technical field, and the knowledge base doesn't cover that. Sin

In [5]:
question = "where is mingora"
answer, latency = cag_query(question, knowledge_base)

print("Answer:", answer)

[QUERY] where is mingora
[SUCCESS] Answer generated in 1.40s
[ANSWER] <think>
Okay, let's see. The user is asking "where is mingora". I need to answer based on the provided knowledge base.

First, I'll look through the knowledge base for any mention of Mingora. In the "Major Cities and Key Destinations" section, there's a subsection about Mingora. It says that Mingora is the center of economic activities and the only urban area of the valley, adjacent to Saidu Sharif. It's located in Swat Valley, which is in Pakistan's Khyber Pakhtunkhwa province. 

The introduction part also mentions that Swat Valley is north of Peshawar. So putting it all together, Mingora is in Swat Valley, next to Saidu Sharif. The answer should include that it's in Pakistan, specifically Khyber Pakhtunkhwa province, and note its proximity to Saidu Sharif and Peshawar. Also, maybe mention it's known for its bazaars and emerald mines. But the main location details are in the Major Cities section. I need to make sure